# Demo Hỏi Đáp trên Tài Liệu bằng LLM

Notebook này xây dựng một pipeline hỏi đáp trên tài liệu theo hướng RAG: người dùng tải tài liệu lên, hệ thống chia nhỏ nội dung, tạo vector embedding, lưu vào cơ sở dữ liệu vector, rồi truy xuất ngữ cảnh liên quan để đưa cho mô hình sinh câu trả lời.


In [ ]:
from google.colab import files
files.upload()

Saving README.txt to README.txt


{'README.txt': b".. -*- mode: rst -*-\n\n|GitHubActions| |Codecov| |CircleCI| |Nightly wheels| |Ruff| |PythonVersion| |PyPI| |DOI| |Benchmark|\n\n\n.. |GitHubActions| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml/badge.svg?\n   :target: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml?query=branch%3Amain\n\n.. |CircleCI| image:: https://circleci.com/gh/scikit-learn/scikit-learn/tree/main.svg?style=shield\n   :target: https://circleci.com/gh/scikit-learn/scikit-learn\n\n.. |Codecov| image:: https://codecov.io/gh/scikit-learn/scikit-learn/branch/main/graph/badge.svg?token=Pk8G9gg3y9\n   :target: https://codecov.io/gh/scikit-learn/scikit-learn\n\n.. |Nightly wheels| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/wheels.yml/badge.svg?event=schedule\n   :target: https://github.com/scikit-learn/scikit-learn/actions?query=workflow%3A%22Wheel+builder%22+event%3Aschedule\n\n.. |Ruff| image:: https://img

## 1. Cài đặt thư viện

Cell này cài các thư viện cần thiết cho toàn bộ pipeline:
- `faiss-cpu`: dùng cho tìm kiếm vector.
- `langchain-google-genai`: hỗ trợ tích hợp các mô hình / thành phần của LangChain.
- `langchain` và `langchain-core`: khung làm việc chính cho xử lý tài liệu, splitter, retriever, chain.


In [ ]:
!pip install faiss-cpu
!pip install -q langchain-google-genai
!pip install -q langchain langchain-core langchain-google-genai

## 2. Tải tài liệu đầu vào

Cell này dùng `TextLoader` để đọc file văn bản `README.txt` từ thư mục làm việc hiện tại.

Kết quả trả về là danh sách `documents`, trong đó mỗi phần tử là một đối tượng tài liệu có thuộc tính `page_content` chứa nội dung văn bản.


In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader(
    "README.txt",
    encoding="utf-8"
)

documents = loader.load()

print(documents[0].page_content[:500])


.. -*- mode: rst -*-

|GitHubActions| |Codecov| |CircleCI| |Nightly wheels| |Ruff| |PythonVersion| |PyPI| |DOI| |Benchmark|


.. |GitHubActions| image:: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml/badge.svg?
   :target: https://github.com/scikit-learn/scikit-learn/actions/workflows/unit-tests.yml?query=branch%3Amain

.. |CircleCI| image:: https://circleci.com/gh/scikit-learn/scikit-learn/tree/main.svg?style=shield
   :target: https://circleci.com/gh/scikit-learn


## 3. Chia tài liệu thành các đoạn nhỏ

Đoạn này dùng `RecursiveCharacterTextSplitter` để cắt văn bản thành các chunk có kích thước phù hợp với mô hình.

- `chunk_size=1000`: mỗi đoạn tối đa khoảng 1000 ký tự.
- `chunk_overlap=150`: các đoạn chồng lấn nhẹ để giữ ngữ cảnh giữa các chunk.
- `separators`: ưu tiên ngắt theo đoạn, dòng, câu, từ để tránh cắt gãy nội dung quá đột ngột.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_documents(documents)

## 4. Tạo embedding cho các chunk

`HuggingFaceEmbeddings` biến mỗi chunk văn bản thành vector số để có thể so sánh độ tương đồng ngữ nghĩa.

Ở đây dùng model `BAAI/bge-m3`, chạy trên CPU và chuẩn hóa vector đầu ra để phù hợp hơn cho truy vấn tương đồng.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name = "BAAI/bge-m3",
    model_kwargs = {'device': 'cpu'},
    encode_kwargs = {'normalize_embeddings': True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## 5. Lưu vector và tạo retriever

Cell này đưa các chunk đã chia vào `QdrantVectorStore`.

- `path="./qdrant3"`: nơi lưu dữ liệu vector trên máy cục bộ.
- `collection_name="scikit-learn"`: tên collection trong Qdrant.
- `as_retriever(search_kwargs={"k": 2})`: tạo bộ truy xuất, mỗi lần tìm sẽ lấy ra 2 đoạn liên quan nhất với câu hỏi.


In [ ]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding = embeddings,
    path = "./qdrant3",
    collection_name = "scikit-learn"
)

retriever = vectorstore.as_retriever(search_kwargs = {"k": 2})



## 6. Truy xuất ngữ cảnh và sinh câu trả lời

Đây là phần chính của pipeline:

1. Khởi tạo mô hình sinh văn bản `TinyLlama/TinyLlama-1.1B-Chat-v1.0`.
2. Đặt câu hỏi mẫu.
3. Dùng `retriever.invoke(question)` để lấy các chunk liên quan nhất.
4. Ghép các chunk đó thành phần `REFERENCE DOCUMENTS`.
5. Tạo prompt yêu cầu mô hình chỉ trả lời dựa trên tài liệu.
6. Gọi model để sinh câu trả lời.
7. In ra đáp án và các nguồn đã được truy xuất.


In [ ]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    do_sample=False
)

# ===============================
# QUESTION
# ===============================
question = "What is scikit-learn?"

# ===============================
# RETRIEVE DATA
# ===============================
retrieved_data = retriever.invoke(question)

source = "\n\n---\n\n".join([doc.page_content for doc in retrieved_data])

# ===============================
# PROMPT
# ===============================
prompt = f"""
You are an intelligent Machine Learning assistant.

Answer the user's question ONLY based on the reference documents below.

If the answer is not found in the documents, reply:
The document does not mention this.

Keep the answer clear, concise, and easy to understand.

REFERENCE DOCUMENTS:
{source}

USER QUESTION:
{question}

ANSWER:
"""

print(prompt)

# ===============================
# GENERATE
# ===============================
response = llm(prompt)

answer = response[0]["generated_text"]

# ===============================
# OUTPUT
# ===============================
print("\n=== ANSWER ===\n")
print(answer)

print("\n=== SOURCES ===\n")

for i, doc in enumerate(retrieved_data):
    print(f"--- Source {i+1} ---")
    print(doc.page_content.strip()[:300] + "...\n")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are an intelligent Machine Learning assistant.

Answer the user's question ONLY based on the reference documents below.

If the answer is not found in the documents, reply:
The document does not mention this.

Keep the answer clear, concise, and easy to understand.

REFERENCE DOCUMENTS:
**scikit-learn** is a Python module for machine learning built on top of
SciPy and is distributed under the 3-Clause BSD license.

The project was started in 2007 by David Cournapeau as a Google Summer
of Code project, and since then many volunteers have contributed. See
the `About us <https://scikit-learn.org/dev/about.html#authors>`__ page
for a list of core contributors.

It is currently maintained by a team of volunteers.

Website: https://scikit-learn.org

Installation
------------

Dependencies
~~~~~~~~~~~~

scikit-learn requires:

- Python (>= |PythonMinVersion|)
- NumPy (>= |NumPyMinVersion|)
- SciPy (>= |SciPyMinVersion|)
- joblib (>= |JoblibMinVersion|)
- threadpoolctl (>= |ThreadpoolctlM

## 7. Kết quả mong đợi

Sau khi chạy toàn bộ notebook, bạn sẽ có:
- một bộ chunk văn bản đã được vector hóa,
- một retriever để tìm đoạn liên quan,
- và một mô hình trả lời câu hỏi dựa trên nội dung tài liệu đầu vào.

Nếu đổi file đầu vào, chỉ cần cập nhật lại `README.txt` hoặc đường dẫn trong cell load tài liệu.
